# Gemini Drive notes → n8n HQ Tasks

Runtime → **Run all**. Sign in with a Save 5 Hours Google account (`@save5hours.ch`, `antubejar96@gmail.com`, `deevlylabs@gmail.com`, or `roman.cajka@gmail.com`).

This creates a real Google Doc (Drive API) and POSTs `{fileId, text}` to `/webhook/public-drive-doc` (retries on boot 404). That path does not need n8n Google userinfo. It only tries `/webhook/meeting-notes-drive` if the public webhook fails, so HQ Tasks are not duplicated.

You do **not** need an n8n login or `WEBHOOK_SECRET`. If the POST succeeds, HQ Tasks get a Drive file ID that is not `inline-*`. If it fails, paste `FILE_URL` into the HQ confirmation task.

Keep `VERIFY_NOTES` in sync with `fixtures/drive-verify-notes.txt`.

In [ ]:
%pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib requests

from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload
import requests

N8N = "https://n8n-production-192e.up.railway.app"
PUBLIC_WEBHOOK = N8N + "/webhook/public-drive-doc"
DRIVE_WEBHOOK = N8N + "/webhook/meeting-notes-drive"
TITLE = "Gemini notes — Drive path verification (n8n)"
VERIFY_NOTES = (
    "Gemini notes — Drive path verification (n8n)\n\n"
    "Attendees: Antoine Bejarano Alvarez, Martin, Roman Cajka.\n\n"
    "Actions agreed:\n"
    "- Antoine will publish the Drive webhook runbook in HQ this week.\n"
    "- Martin will review HQ Tasks with Origin Meeting after the Drive file lands.\n"
    "- Roman will confirm the Meet Recordings folder URL on the Drive confirmation task.\n"
)

creds, _ = google.auth.default()
if creds and creds.expired and getattr(creds, "refresh_token", None):
    creds.refresh(Request())
token = getattr(creds, "token", None) or ""
if not token:
    raise SystemExit("No Google access token. Re-run after signing in.")

drive = build("drive", "v3", credentials=creds)
file_id = ""
file_url = ""
try:
    media = MediaInMemoryUpload(VERIFY_NOTES.encode("utf-8"), mimetype="text/plain", resumable=False)
    created = drive.files().create(
        body={"name": TITLE, "mimeType": "application/vnd.google-apps.document"},
        media_body=media,
        fields="id,webViewLink",
        supportsAllDrives=True,
    ).execute()
    file_id = created["id"]
    file_url = created.get("webViewLink") or ""
except Exception as err:
    print("Drive convert failed, using Docs API", err)
    docs = build("docs", "v1", credentials=creds)
    created_doc = docs.documents().create(body={"title": TITLE}).execute()
    file_id = created_doc["documentId"]
    docs.documents().batchUpdate(
        documentId=file_id,
        body={"requests": [{"insertText": {"location": {"index": 1}, "text": VERIFY_NOTES}}]},
    ).execute()
file_url = file_url or f"https://docs.google.com/document/d/{file_id}/edit"
try:
    drive.permissions().create(
        fileId=file_id,
        body={"type": "anyone", "role": "reader"},
        supportsAllDrives=True,
    ).execute()
except Exception as err:
    print("share skipped", err)

payload = {
    "fileId": file_id,
    "url": file_url,
    "name": TITLE,
    "mimeType": "application/vnd.google-apps.document",
    "webViewLink": file_url,
    "text": VERIFY_NOTES,
}

import time

def post_with_retry(url, body, timeout=120, attempts=6):
    last = None
    for attempt in range(1, attempts + 1):
        last = requests.post(url, json=body, timeout=timeout)
        text = last.text or ""
        if 200 <= last.status_code < 300:
            return last
        retryable = last.status_code in (404, 502, 503) or "not registered" in text.lower()
        if not retryable or attempt >= attempts:
            return last
        time.sleep(10)
    return last

public = post_with_retry(PUBLIC_WEBHOOK, payload)
print("public-drive-doc HTTP", public.status_code)
print(public.text[:500])

drive_ok = False
if public.status_code >= 300 and token:
    drive_resp = post_with_retry(DRIVE_WEBHOOK, {**payload, "googleAccessToken": token})
    print("meeting-notes-drive HTTP", drive_resp.status_code)
    print(drive_resp.text[:500])
    drive_ok = drive_resp.status_code < 300

print("FILE_ID", file_id)
print("FILE_URL", file_url)
print("If HQ Tasks stay empty, paste FILE_URL into https://app.notion.com/p/3cd0b26fcc4e819bb9ead19d74fb64a6")
if public.status_code >= 300 and not drive_ok:
    public.raise_for_status()